<a href="https://colab.research.google.com/github/rafiebazine/rag-image-captioning/blob/hadil/phase_02_preprocessing/03_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!git clone https://github.com/rafiebazine/rag-image-captioning.git
%cd rag-image-captioning
!git checkout hadil
!pip -q install transformers

from google.colab import drive
drive.mount("/content/drive")

Cloning into 'rag-image-captioning'...
remote: Enumerating objects: 191, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 191 (delta 23), reused 4 (delta 2), pack-reused 133 (from 1)
Receiving objects: 100% (191/191), 4.29 MiB | 6.50 MiB/s, done.
Resolving deltas: 100% (81/81), done.
/content/rag-image-captioning/rag-image-captioning
branch 'hadil' set up to track 'origin/hadil'.
Switched to a new branch 'hadil'
Mounted at /content/drive


In [3]:
%cd /content/rag-image-captioning
!git status | head -3

/content/rag-image-captioning
On branch hadil
Your branch is up to date with 'origin/hadil'.



In [4]:
import json, os, time, requests
from concurrent.futures import ThreadPoolExecutor

PROC = "phase_02_preprocessing/processed"
DRIVE_DIR = "/content/drive/MyDrive/rag_embeddings"
IMG_DIR = "/content/images"
os.makedirs(DRIVE_DIR, exist_ok=True); os.makedirs(IMG_DIR, exist_ok=True)

MODEL_NAME = "openai/clip-vit-base-patch32"
BATCH = 128

splits = {n: json.load(open(f"{PROC}/{n}_clean.json")) for n in ["kb_10k","val_100","test_100","test_1000"]}
src = {"kb_10k":"train2017","val_100":"train2017","test_100":"val2017","test_1000":"val2017"}

def download(args):
    folder, fname = args
    path = f"{IMG_DIR}/{fname}"
    if os.path.exists(path): return True
    try:
        r = requests.get(f"http://images.cocodataset.org/{folder}/{fname}", timeout=30)
        r.raise_for_status(); open(path,"wb").write(r.content); return True
    except Exception as e:
        return False

jobs = [(src[n], x["file_name"]) for n, d in splits.items() for x in d]
with ThreadPoolExecutor(16) as ex:
    ok = list(ex.map(download, jobs))
print("downloaded:", sum(ok), "/", len(jobs))

downloaded: 11200 / 11200


In [ ]:
import torch, numpy as np
from PIL import Image, ImageOps
from transformers import CLIPModel, CLIPProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
model = CLIPModel.from_pretrained(MODEL_NAME).to(device).eval()
proc = CLIPProcessor.from_pretrained(MODEL_NAME)

def load_img(fname):
    im = Image.open(f"{IMG_DIR}/{fname}")
    return ImageOps.exif_transpose(im).convert("RGB")      # EXIF orientation + RGB

def l2(t):
    return torch.nn.functional.normalize(t, dim=-1).cpu().numpy()

@torch.no_grad()
def embed_images(files):
    out = []
    for i in range(0, len(files), BATCH):
        ims = [load_img(f) for f in files[i:i+BATCH]]
        x = proc(images=ims, return_tensors="pt").to(device)
        pooled = model.vision_model(pixel_values=x["pixel_values"]).pooler_output
        out.append(l2(model.visual_projection(pooled)))
    return np.concatenate(out).astype("float32")

@torch.no_grad()
def embed_texts(texts):
    out = []
    for i in range(0, len(texts), 512):
        x = proc(text=texts[i:i+512], return_tensors="pt", padding=True, truncation=True, max_length=77).to(device)
        pooled = model.text_model(input_ids=x["input_ids"], attention_mask=x["attention_mask"]).pooler_output
        out.append(l2(model.text_projection(pooled)))
    return np.concatenate(out).astype("float32")

t0 = time.time()
for n, d in splits.items():
    emb = embed_images([x["file_name"] for x in d])
    np.save(f"{DRIVE_DIR}/{n}_image_emb.npy", emb)
    json.dump([x["image_id"] for x in d], open(f"{DRIVE_DIR}/{n}_image_ids.json", "w"))
    print(n, emb.shape)

kb = splits["kb_10k"]
texts, owners = [], []
for x in kb:
    for j, c in enumerate(x["captions_clean"]):
        texts.append(c); owners.append((x["image_id"], j))
txt_emb = embed_texts(texts)
np.save(f"{DRIVE_DIR}/kb_text_emb.npy", txt_emb)
json.dump(owners, open(f"{DRIVE_DIR}/kb_text_owners.json", "w"))
print("kb text", txt_emb.shape, "| time", round(time.time() - t0), "s")

device: cuda


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

kb_10k (10000, 512)
val_100 (100, 512)
test_100 (100, 512)
test_1000 (1000, 512)


In [2]:
import os, json, time
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/rag_embeddings"
for f in sorted(os.listdir(DRIVE_DIR)):
    print(f, round(os.path.getsize(f"{DRIVE_DIR}/{f}") / 1e6, 1), "MB")

Mounted at /content/drive
kb_10k_image_emb.npy 20.5 MB
kb_10k_image_ids.json 0.1 MB
kb_text_emb.npy 102.4 MB
kb_text_owners.json 0.6 MB
test_1000_image_emb.npy 2.0 MB
test_1000_image_ids.json 0.0 MB
test_100_image_emb.npy 0.2 MB
test_100_image_ids.json 0.0 MB
val_100_image_emb.npy 0.2 MB
val_100_image_ids.json 0.0 MB


In [3]:
import os, json, numpy as np
from google.colab import drive
drive.mount("/content/drive")

!rm -rf /content/rag-image-captioning
!git clone -q https://github.com/rafiebazine/rag-image-captioning.git /content/rag-image-captioning
%cd /content/rag-image-captioning
!git checkout -q hadil

DRIVE_DIR = "/content/drive/MyDrive/rag_embeddings"
PROC = "phase_02_preprocessing/processed"
names = ["kb_10k", "val_100", "test_100", "test_1000"]

# 1) shapes, NaN, norms
emb, ids = {}, {}
for n in names:
    emb[n] = np.load(f"{DRIVE_DIR}/{n}_image_emb.npy")
    ids[n] = json.load(open(f"{DRIVE_DIR}/{n}_image_ids.json"))
    assert emb[n].shape == (len(ids[n]), 512), n
    assert not np.isnan(emb[n]).any(), n
    assert len(set(ids[n])) == len(ids[n]), f"duplicate ids in {n}"
    print(n, emb[n].shape, "mean norm:", round(float(np.linalg.norm(emb[n], axis=1).mean()), 3))

txt = np.load(f"{DRIVE_DIR}/kb_text_emb.npy")
owners = json.load(open(f"{DRIVE_DIR}/kb_text_owners.json"))
assert txt.shape == (50000, 512) and len(owners) == 50000 and not np.isnan(txt).any()
assert {o[0] for o in owners} == set(ids["kb_10k"])
print("kb_text", txt.shape, "mean norm:", round(float(np.linalg.norm(txt, axis=1).mean()), 3))

# 2) leakage audit:
kb_set = set(ids["kb_10k"])
for n in ["val_100", "test_100", "test_1000"]:
    assert not (kb_set & set(ids[n])), f"LEAK between kb_10k and {n}"
print("leakage audit: OK (no overlap between KB and val/test)")

# 3) sanity retrieval اinsaidKB: caption -> image
rng = np.random.default_rng(42)
sample = rng.choice(len(txt), 2000, replace=False)
sims = txt[sample] @ emb["kb_10k"].T
order = np.argsort(-sims, axis=1)[:, :10]
kb_ids = np.array(ids["kb_10k"])
true_ids = np.array([owners[i][0] for i in sample])
for k in [1, 5, 10]:
    hit = (kb_ids[order[:, :k]] == true_ids[:, None]).any(axis=1).mean()
    print(f"text->image Recall@{k} (KB sanity, 2000 captions): {hit:.3f}")

# 4) exp: val image -> near to caption in KB
kb = json.load(open(f"{PROC}/kb_10k_clean.json"))
cap = {(x["image_id"], j): c for x in kb for j, c in enumerate(x["captions_raw"])}
val = json.load(open(f"{PROC}/val_100_clean.json"))
s = emb["val_100"] @ txt.T
for i in range(3):
    j = int(s[i].argmax())
    print("\nGT   :", val[i]["captions_raw"][0].strip())
    print("RETR :", cap[tuple(owners[j])].strip(), f"(score {s[i, j]:.2f})")

# 5)
info = {"model": "openai/clip-vit-base-patch32", "dim": 512, "image_batch": 128, "text_batch": 512,
        "normalisation": "L2", "dtype": "float32", "text_source": "captions_clean (KB only)",
        "n_image_vectors_kb": len(ids["kb_10k"]), "n_text_vectors_kb": len(owners),
        "image_preprocessing": "EXIF transpose, RGB, CLIPProcessor resize/normalise", "device": "cuda (T4)"}
json.dump(info, open(f"{PROC}/embedding_info.json", "w"), indent=1)
print("\nsaved embedding_info.json")

from google.colab import files
files.download(f"{PROC}/embedding_info.json")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/rag-image-captioning
kb_10k (10000, 512) mean norm: 1.0
val_100 (100, 512) mean norm: 1.0
test_100 (100, 512) mean norm: 1.0
test_1000 (1000, 512) mean norm: 1.0
kb_text (50000, 512) mean norm: 1.0
leakage audit: OK (no overlap between KB and val/test)
text->image Recall@1 (KB sanity, 2000 captions): 0.182
text->image Recall@5 (KB sanity, 2000 captions): 0.359
text->image Recall@10 (KB sanity, 2000 captions): 0.458

GT   : It looks like there is writing on the bottom of the Stop sign.
RETR : A stop sign that has been graffitied to say never stop loving. (score 0.34)

GT   : A city street filled with traffic and buses.
RETR : a motor bike and other cars sit parked at an intersection (score 0.35)

GT   : Two birds sitting on top of a wooden table near tinfoil.
RETR : A bird is eating bread from a basket on a table. (score 0.32)

saved embedding_info.js

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>